In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip -q install ijson
import os, json, zlib, random, ijson, shutil
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Exploration-Lab/CS-781-Capstone"
RAW = "/kaggle/working/raw"
OUT = "/kaggle/working/data"
os.makedirs(OUT, exist_ok=True)

def norm(t): return " ".join(t.split())

def build(path, source, keep_prob=1.0, val_pct=2, seed=0):
    rng = random.Random(seed)
    seen = ntr = nva = 0
    with open(f"{OUT}/{source}_train.jsonl", "w", encoding="utf-8") as tr, \
         open(f"{OUT}/{source}_val.jsonl", "w", encoding="utf-8") as va, \
         open(path, "rb") as f:
        for i, rec in enumerate(ijson.items(f, "item")):
            seen += 1
            text = norm(rec.get("text") or "")
            if not text:
                continue
            if zlib.crc32(f"{source}:{i}".encode()) % 100 < val_pct:
                va.write(json.dumps({"text": text}) + "\n"); nva += 1
            elif rng.random() < keep_prob:
                tr.write(json.dumps({"text": text}) + "\n"); ntr += 1
    print(f"{source}: seen {seen}, train {ntr}, val {nva}")

jobs = [("train_acts.json", "acts", 1.0),
        ("train_supreme_court.json", "sc", 1.0),
        ("train_tribunals.json", "tribunals", 0.1)]

for fname, source, keep in jobs:
    p = hf_hub_download(REPO, f"Phase_1_Pretraining/Train/{fname}",
                        repo_type="dataset", token=token, local_dir=RAW)
    build(p, source, keep_prob=keep)
    os.remove(p)

t = hf_hub_download(REPO, "Phase_1_Pretraining/Test/test.json",
                    repo_type="dataset", token=token, local_dir=RAW)
shutil.copy(t, f"{OUT}/test.json")
shutil.rmtree(RAW, ignore_errors=True)
print(os.listdir(OUT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.9 MB/s eta 0:00:00


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

acts: seen 18826, train 18455, val 367


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

sc: seen 55549, train 54443, val 1085


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

tribunals: seen 869132, train 85084, val 17132


test.json: 0.00B [00:00, ?B/s]

['tribunals_val.jsonl', 'sc_train.jsonl', 'tribunals_train.jsonl', 'acts_train.jsonl', 'sc_val.jsonl', 'acts_val.jsonl', 'test.json']
